In [1]:
# ==========================================================
# Task 1: Web Scraping (TechReads Retail Analytics)
# Source: Goodreads "data-engineering" shelf
# Steps:
# 1) Collect book URLs from shelf page(s)
# 2) Visit each book page to extract Title, Author, Year, Rating, Price
# 3) Save to techreads_books.csv
# ==========================================================

import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Use a browser-like User-Agent to reduce blocking.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
}

def fetch_html(url: str, timeout: int = 20) -> str:
    """Fetch HTML safely with headers."""
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.encoding = "utf-8"
    return resp.text, resp.status_code

def looks_blocked(html: str) -> bool:
    """Basic detection for CAPTCHA/robot blocks."""
    t = html.lower()
    return ("captcha" in t) or ("robot" in t and "check" in t) or ("automated access" in t)

def extract_year(text: str) -> str:
    """Extract first 4-digit year from text."""
    m = re.search(r"\b(19|20)\d{2}\b", text)
    return m.group(0) if m else "N/A"

def extract_price(text: str) -> str:
    """
    Extract a price if present on the page (e.g., 'Kindle $42.99' or '£31.50').
    Returns 'N/A' if none found.
    """
    # Capture £xx.xx or $xx.xx
    m = re.search(r"([£$])\s?(\d+(?:\.\d{2})?)", text)
    if m:
        return f"{m.group(1)}{m.group(2)}"
    return "N/A"

print("Helpers ready.")

Helpers ready.


In [3]:
# Goodreads shelf page (you can change this URL if needed)
SHELF_URL = "https://www.goodreads.com/shelf/show/data-engineering"

TARGET_BOOKS = 15          # minimum required
MAX_PAGES = 5              # keep low to avoid blocks
POLITE_DELAY = 1.0         # seconds between requests

book_urls = []
seen = set()

for page in range(1, MAX_PAGES + 1):
    url = SHELF_URL if page == 1 else f"{SHELF_URL}?page={page}"
    html, status = fetch_html(url)

    if status != 200:
        print(f"Shelf page {page} failed: HTTP {status}")
        break
    if looks_blocked(html):
        print("Blocked/CAPTCHA detected on shelf page. Try again later or reduce requests.")
        break

    soup = BeautifulSoup(html, "html.parser")

    # Find links to book pages
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/book/show/" in href:
            full = urljoin("https://www.goodreads.com", href.split("?")[0])
            if full not in seen:
                seen.add(full)
                book_urls.append(full)

        if len(book_urls) >= TARGET_BOOKS:
            break

    print(f"Shelf page {page}: collected {len(book_urls)} book URLs so far")

    if len(book_urls) >= TARGET_BOOKS:
        break

    time.sleep(POLITE_DELAY)

print("\nTotal book URLs collected:", len(book_urls))
book_urls[:5]

Shelf page 1: collected 15 book URLs so far

Total book URLs collected: 15


['https://www.goodreads.com/book/show/61218623-fundamentals-of-data-engineering',
 'https://www.goodreads.com/book/show/23463279-designing-data-intensive-applications',
 'https://www.goodreads.com/book/show/748203.The_Data_Warehouse_Toolkit',
 'https://www.goodreads.com/book/show/55841851-data-pipelines-pocket-reference',
 'https://www.goodreads.com/book/show/44647144-database-internals']

In [4]:
# Visit each book page to extract: Title, Author, Year, Rating, RatingsCount
rows = []

for i, url in enumerate(book_urls, start=1):
    html, status = fetch_html(url)
    if status != 200:
        print(f"[{i}] Failed: HTTP {status} -> {url}")
        continue
    if looks_blocked(html):
        print(f"[{i}] Blocked/CAPTCHA detected -> stopping early")
        break

    soup = BeautifulSoup(html, "html.parser")
    page_text = soup.get_text(" ", strip=True)

    # Title
    title = "N/A"
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(" ", strip=True)

    # Author (new Goodreads UI)
    author = "N/A"
    author_span = soup.select_one('span.ContributorLink__name[data-testid="name"]')
    if author_span and author_span.get_text(strip=True):
        author = author_span.get_text(" ", strip=True)
    else:
        author_link = soup.find("a", href=re.compile(r"^/author/show/"))
        if author_link:
            author = author_link.get_text(" ", strip=True)

    # Year
    year = extract_year(page_text)

    # Rating (average rating)
    rating = "N/A"
    rating_candidates = re.findall(r"\b([0-5]\.\d{1,2})\b", page_text)
    if rating_candidates:
        rating = float(rating_candidates[0])

    # Ratings count (popularity indicator – useful for "trending")
    # Example text on page: "10,687 ratings 947 reviews"
    ratings_count = "N/A"
    m_rc = re.search(r"([\d,]+)\s+ratings", page_text.lower())
    if m_rc:
        ratings_count = int(m_rc.group(1).replace(",", ""))

    # Optional: Reviews count
    reviews_count = "N/A"
    m_rev = re.search(r"([\d,]+)\s+reviews", page_text.lower())
    if m_rev:
        reviews_count = int(m_rev.group(1).replace(",", ""))

    rows.append({
        "Title": title,
        "Author": author,
        "Year": year,
        "Rating": rating,
        "RatingsCount": ratings_count,   # ✅ replaces Price
        "ReviewsCount": reviews_count,   # optional extra field
        "Source": "goodreads",
        "URL": url
    })

    print(f"[{i}] OK: {title[:55]} | {author} | rating={rating} | ratings={ratings_count}")
    time.sleep(POLITE_DELAY)

df = pd.DataFrame(rows)
print("\nRows extracted:", len(df))
df.head(15)

[1] OK: Fundamentals of Data Engineering: Plan and Build Robust | Joe Reis | rating=4.18 | ratings=929
[2] OK: Designing Data-Intensive Applications | Martin Kleppmann | rating=4.7 | ratings=10714
[3] OK: The Data Warehouse Toolkit: The Complete Guide to Dimen | Ralph Kimball | rating=4.19 | ratings=1037
[4] OK: Data Pipelines Pocket Reference: Moving and Processing  | James Densmore | rating=3.77 | ratings=241
[5] OK: Database Internals: A deep-dive into how distributed da | Alex Petrov | rating=4.26 | ratings=561
[6] OK: Data Mesh: Delivering Data-Driven Value at Scale | Zhamak Dehghani | rating=3.79 | ratings=380
[7] OK: Data Pipelines with Apache Airflow | Bas P. Harenslak | rating=4.31 | ratings=118
[8] OK: Spark: The Definitive Guide: Big Data Processing Made S | Bill Chambers | rating=4.14 | ratings=283
[9] OK: Kafka: The Definitive Guide: Real-Time Data and Stream  | Neha Narkhede | rating=4.15 | ratings=728
[10] OK: Making Sense of Stream Processing | Martin Kleppmann | rating

,Title,Author,Year,Rating,RatingsCount,ReviewsCount,Source,URL
0,Fundamentals of Data Engineering: Plan and Bui...,Joe Reis,2022,4.18,929,97,goodreads,https://www.goodreads.com/book/show/61218623-f...
1,Designing Data-Intensive Applications,Martin Kleppmann,2015,4.70,10714,949,goodreads,https://www.goodreads.com/book/show/23463279-d...
2,The Data Warehouse Toolkit: The Complete Guide...,Ralph Kimball,1996,4.19,1037,65,goodreads,https://www.goodreads.com/book/show/748203.The...
3,Data Pipelines Pocket Reference: Moving and Pr...,James Densmore,2021,3.77,241,31,goodreads,https://www.goodreads.com/book/show/55841851-d...
4,Database Internals: A deep-dive into how distr...,Alex Petrov,2019,4.26,561,70,goodreads,https://www.goodreads.com/book/show/44647144-d...
5,Data Mesh: Delivering Data-Driven Value at Scale,Zhamak Dehghani,2022,3.79,380,40,goodreads,https://www.goodreads.com/book/show/58230358-d...
6,Data Pipelines with Apache Airflow,Bas P. Harenslak,2020,4.31,118,16,goodreads,https://www.goodreads.com/book/show/52778122-d...
7,Spark: The Definitive Guide: Big Data Processi...,Bill Chambers,2018,4.14,283,31,goodreads,https://www.goodreads.com/book/show/38467996-s...
8,Kafka: The Definitive Guide: Real-Time Data an...,Neha Narkhede,2017,4.15,728,81,goodreads,https://www.goodreads.com/book/show/28321010-k...
9,Making Sense of Stream Processing,Martin Kleppmann,2016,4.25,190,30,goodreads,https://www.goodreads.com/book/show/29598815-m...


In [5]:
# Save CSV with 5+ fields (Price replaced by RatingsCount)
df_out = df[["Title", "Author", "Year", "Rating", "RatingsCount", "ReviewsCount", "Source", "URL"]].copy()

df_out.to_csv("techreads_books.csv", index=False, encoding="utf-8")
print("Saved: techreads_books.csv")

df_out.head(15)

Saved: techreads_books.csv


,Title,Author,Year,Rating,RatingsCount,ReviewsCount,Source,URL
0,Fundamentals of Data Engineering: Plan and Bui...,Joe Reis,2022,4.18,929,97,goodreads,https://www.goodreads.com/book/show/61218623-f...
1,Designing Data-Intensive Applications,Martin Kleppmann,2015,4.70,10714,949,goodreads,https://www.goodreads.com/book/show/23463279-d...
2,The Data Warehouse Toolkit: The Complete Guide...,Ralph Kimball,1996,4.19,1037,65,goodreads,https://www.goodreads.com/book/show/748203.The...
3,Data Pipelines Pocket Reference: Moving and Pr...,James Densmore,2021,3.77,241,31,goodreads,https://www.goodreads.com/book/show/55841851-d...
4,Database Internals: A deep-dive into how distr...,Alex Petrov,2019,4.26,561,70,goodreads,https://www.goodreads.com/book/show/44647144-d...
5,Data Mesh: Delivering Data-Driven Value at Scale,Zhamak Dehghani,2022,3.79,380,40,goodreads,https://www.goodreads.com/book/show/58230358-d...
6,Data Pipelines with Apache Airflow,Bas P. Harenslak,2020,4.31,118,16,goodreads,https://www.goodreads.com/book/show/52778122-d...
7,Spark: The Definitive Guide: Big Data Processi...,Bill Chambers,2018,4.14,283,31,goodreads,https://www.goodreads.com/book/show/38467996-s...
8,Kafka: The Definitive Guide: Real-Time Data an...,Neha Narkhede,2017,4.15,728,81,goodreads,https://www.goodreads.com/book/show/28321010-k...
9,Making Sense of Stream Processing,Martin Kleppmann,2016,4.25,190,30,goodreads,https://www.goodreads.com/book/show/29598815-m...


# Task 2: MySQL Database Pipeline – TechReads Retail Analytics
**Module:** CMP-X304-0 Data Engineering  
**Database:** techreads_db  
**Table:** books  
**Input:** techreads_books.csv

## Step 1: Import Required Libraries

In [6]:
import mysql.connector
import pandas as pd

## Step 2: Load and Inspect CSV Data

In [8]:
df = pd.read_csv('techreads_books.csv',
                 encoding='utf-8-sig',
                 na_values=['N/A', 'NA', 'n/a', ''])

print(f'Total rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print('\nData types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

Total rows: 15
Columns: ['Title', 'Author', 'Year', 'Rating', 'RatingsCount', 'ReviewsCount', 'Source', 'URL']

Data types:
Title               str
Author              str
Year              int64
Rating          float64
RatingsCount      int64
ReviewsCount      int64
Source              str
URL                 str
dtype: object

First 5 rows:


,Title,Author,Year,Rating,RatingsCount,ReviewsCount,Source,URL
0,Fundamentals of Data Engineering: Plan and Bui...,Joe Reis,2022,4.18,929,97,goodreads,https://www.goodreads.com/book/show/61218623-f...
1,Designing Data-Intensive Applications,Martin Kleppmann,2015,4.70,10714,949,goodreads,https://www.goodreads.com/book/show/23463279-d...
2,The Data Warehouse Toolkit: The Complete Guide...,Ralph Kimball,1996,4.19,1037,65,goodreads,https://www.goodreads.com/book/show/748203.The...
3,Data Pipelines Pocket Reference: Moving and Pr...,James Densmore,2021,3.77,241,31,goodreads,https://www.goodreads.com/book/show/55841851-d...
4,Database Internals: A deep-dive into how distr...,Alex Petrov,2019,4.26,561,70,goodreads,https://www.goodreads.com/book/show/44647144-d...


## Step 3: Connect to MySQL and Create Database

In [9]:
conn = mysql.connector.connect(
    host='localhost',
    port=8868,
    user='root',
    password='123123'  
)
cursor = conn.cursor()

cursor.execute('CREATE DATABASE IF NOT EXISTS techreads_db')
print('Database techreads_db created (or already exists).')

cursor.execute('USE techreads_db')
print('Using database: techreads_db')

Database techreads_db created (or already exists).
Using database: techreads_db


## Step 4: Create the Books Table

In [10]:
cursor.execute('DROP TABLE IF EXISTS books')

create_table_query = '''
CREATE TABLE books (
    id             INT AUTO_INCREMENT PRIMARY KEY,
    title          VARCHAR(255) NOT NULL,
    author         VARCHAR(255),
    year           VARCHAR(10),
    rating         DECIMAL(3,2),
    ratings_count  INT,
    reviews_count  INT,
    source         VARCHAR(50),
    url            VARCHAR(500)
)
''';
cursor.execute(create_table_query)
print('Table books created successfully.')

cursor.execute('DESCRIBE books')
print('\nTable structure:')
for row in cursor.fetchall():
    print(row)

Table books created successfully.

Table structure:
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('title', 'varchar(255)', 'NO', '', None, '')
('author', 'varchar(255)', 'YES', '', None, '')
('year', 'varchar(10)', 'YES', '', None, '')
('rating', 'decimal(3,2)', 'YES', '', None, '')
('ratings_count', 'int', 'YES', '', None, '')
('reviews_count', 'int', 'YES', '', None, '')
('source', 'varchar(50)', 'YES', '', None, '')
('url', 'varchar(500)', 'YES', '', None, '')


## Step 5: Import CSV Data into MySQL

In [11]:
insert_query = '''
    INSERT INTO books (title, author, year, rating, ratings_count, reviews_count, source, url)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
''';

inserted = 0
for _, row in df.iterrows():
    title = row['Title'] if not pd.isna(row['Title']) else None
    author = row['Author'] if not pd.isna(row['Author']) else None
    year = str(int(row['Year'])) if not pd.isna(row['Year']) else None
    rating = float(row['Rating']) if not pd.isna(row['Rating']) else None
    ratings_count = int(row['RatingsCount']) if not pd.isna(row['RatingsCount']) else None
    reviews_count = int(row['ReviewsCount']) if not pd.isna(row['ReviewsCount']) else None
    source = row['Source'] if not pd.isna(row['Source']) else None
    url = row['URL'] if not pd.isna(row['URL']) else None
    
    cursor.execute(insert_query, (
        title,
        author,
        year,
        rating,
        ratings_count,
        reviews_count,
        source,
        url
    ))
    inserted += 1

conn.commit()
print(f'Successfully inserted {inserted} records into books table.')

Successfully inserted 15 records into books table.


## Step 6: SQL Queries

In [12]:
query1 = '''
    SELECT title, rating, ratings_count
    FROM books
    ORDER BY rating DESC
''';
cursor.execute(query1)
results1 = cursor.fetchall()

print('Query 1: All books ordered by rating (high to low)')
print(f'{"Title":<70} {"Rating":<10} {"RatingsCount"}')
print('-' * 95)
for row in results1:
    print(f'{str(row[0])[:67]:<70} {str(row[1]):<10} {row[2]}')

Query 1: All books ordered by rating (high to low)
Title                                                                  Rating     RatingsCount
-----------------------------------------------------------------------------------------------
Designing Data-Intensive Applications                                  4.70       10714
Data Pipelines with Apache Airflow                                     4.31       118
Learning Spark: Lightning-Fast Data Analytics                          4.30       152
Database Internals: A deep-dive into how distributed data systems w    4.26       561
Making Sense of Stream Processing                                      4.25       190
Data Engineering with AWS: Learn how to design and build cloud-base    4.21       19
The Data Warehouse Toolkit: The Complete Guide to Dimensional Model    4.19       1037
Architecting Modern Data Platforms: A Guide to Enterprise Hadoop at    4.19       26
Fundamentals of Data Engineering: Plan and Build Robust Data System  

In [13]:
query2 = '''
    SELECT title, author, rating, reviews_count
    FROM books
    WHERE rating >= 4.0
    ORDER BY rating DESC
''';
cursor.execute(query2)
results2 = cursor.fetchall()

print('Query 2: Books with rating >= 4.0')
print(f'{"Title":<55} {"Author":<20} {"Rating":<10} {"Reviews"}')
print('-' * 100)
for row in results2:
    print(f'{str(row[0])[:52]:<55} {str(row[1])[:17]:<20} {str(row[2]):<10} {row[3]}')

Query 2: Books with rating >= 4.0
Title                                                   Author               Rating     Reviews
----------------------------------------------------------------------------------------------------
Designing Data-Intensive Applications                   Martin Kleppmann     4.70       949
Data Pipelines with Apache Airflow                      Bas P. Harenslak     4.31       16
Learning Spark: Lightning-Fast Data Analytics           Jules S. Damji       4.30       12
Database Internals: A deep-dive into how distributed    Alex Petrov          4.26       70
Making Sense of Stream Processing                       Martin Kleppmann     4.25       30
Data Engineering with AWS: Learn how to design and b    Gareth Eagar         4.21       2
The Data Warehouse Toolkit: The Complete Guide to Di    Ralph Kimball        4.19       65
Architecting Modern Data Platforms: A Guide to Enter    Jan Kunigk           4.19       8
Fundamentals of Data Engineering: Plan and

In [14]:
query3 = '''
    SELECT title, author, ratings_count, reviews_count
    FROM books
    ORDER BY ratings_count DESC
    LIMIT 5
''';
cursor.execute(query3)
results3 = cursor.fetchall()

print('Query 3: Top 5 most rated books')
print(f'{"Title":<55} {"Author":<20} {"Ratings":<10} {"Reviews"}')
print('-' * 100)
for row in results3:
    print(f'{str(row[0])[:52]:<55} {str(row[1])[:17]:<20} {str(row[2]):<10} {row[3]}')

Query 3: Top 5 most rated books
Title                                                   Author               Ratings    Reviews
----------------------------------------------------------------------------------------------------
Designing Data-Intensive Applications                   Martin Kleppmann     10714      949
The Data Warehouse Toolkit: The Complete Guide to Di    Ralph Kimball        1037       65
Fundamentals of Data Engineering: Plan and Build Rob    Joe Reis             929        97
Kafka: The Definitive Guide: Real-Time Data and Stre    Neha Narkhede        728        81
Database Internals: A deep-dive into how distributed    Alex Petrov          561        70


## Step 7: Close Connection

In [15]:
cursor.close()
conn.close()
print('MySQL connection closed.')

MySQL connection closed.


# Task 4: MongoDB Integration & Performance Comparison

In [16]:
import pymongo
import pandas as pd
import os
import json
import time

print("Libraries imported successfully!")

Libraries imported successfully!


In [17]:
import os
import pandas as pd
import json

file_path = r'D:\Trae\DECW1---TechReads-Retail-Analytics\Task 3\Output\techreads_books_output.csv'

df = pd.read_csv(file_path, 
                 encoding='utf-8-sig',
                 na_values=['N/A', 'NA', 'n/a', ''])

# Convert to JSON
books_json = json.loads(df.to_json(orient='records'))

print(f"Total records converted to JSON: {len(books_json)}")
print("\nFirst record:")
print(json.dumps(books_json[0], indent=2))

Total records converted to JSON: 15

First record:
{
  "title": "Fundamentals of Data Engineering: Plan and Build Robust Data Systems",
  "author": "Joe Reis",
  "year": 2022,
  "rating": 4.18,
  "ratings_count": 929,
  "reviews_count": 97
}


In [18]:
import pymongo

# Connect to MongoDB
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["techreads_db"]
collection = db["books"]

# Drop existing collection to start fresh
collection.drop()
print("Old collection dropped (if existed).")

# Insert all books
result = collection.insert_many(books_json)
print(f"Successfully inserted {len(result.inserted_ids)} books into MongoDB!")

# Verify
print(f"\nTotal documents in collection: {collection.count_documents({})}")

Old collection dropped (if existed).
Successfully inserted 15 books into MongoDB!

Total documents in collection: 15


In [19]:
import time

# Query 1: Books with rating >= 4.0
start = time.time()

results = collection.find(
    {"Rating": {"$gte": 4.0}},
    {"_id": 0, "Title": 1, "Author": 1, "Rating": 1}
).sort("Rating", -1)

end = time.time()
mongo_time_q1 = (end - start) * 1000

print("Query 1: Books with Rating >= 4.0")
print(f"{'Title':<55} {'Author':<20} {'Rating'}")
print('-' * 85)
for book in results:
    print(f"{str(book['Title'])[:52]:<55} {str(book['Author'])[:17]:<20} {book['Rating']}")

print(f"\nMongoDB Query Time: {mongo_time_q1:.4f} ms")

Query 1: Books with Rating >= 4.0
Title                                                   Author               Rating
-------------------------------------------------------------------------------------

MongoDB Query Time: 0.0000 ms


In [20]:
# Query 2: Books published after 2019
start = time.time()

results = collection.find(
    {"Year": {"$gt": 2019}},
    {"_id": 0, "Title": 1, "Author": 1, "Year": 1, "Rating": 1}
).sort("Year", -1)

end = time.time()
mongo_time_q2 = (end - start) * 1000

print("Query 2: Books published after 2019")
print(f"{'Title':<55} {'Author':<20} {'Year':<10} {'Rating'}")
print('-' * 90)
for book in results:
    print(f"{str(book['Title'])[:52]:<55} {str(book['Author'])[:17]:<20} {str(book['Year']):<10} {book['Rating']}")

print(f"\nMongoDB Query Time: {mongo_time_q2:.4f} ms")

Query 2: Books published after 2019
Title                                                   Author               Year       Rating
------------------------------------------------------------------------------------------

MongoDB Query Time: 0.0000 ms


In [21]:
# Query 3: Books with more than 500 ratings (popular books)
start = time.time()

results = collection.find(
    {"RatingsCount": {"$gt": 500}},
    {"_id": 0, "Title": 1, "Author": 1, "RatingsCount": 1, "Rating": 1}
).sort("RatingsCount", -1)

end = time.time()
mongo_time_q3 = (end - start) * 1000

print("Query 3: Books with more than 500 ratings")
print(f"{'Title':<55} {'Author':<20} {'RatingsCount':<15} {'Rating'}")
print('-' * 95)
for book in results:
    print(f"{str(book['Title'])[:52]:<55} {str(book['Author'])[:17]:<20} {str(book['RatingsCount']):<15} {book['Rating']}")

print(f"\nMongoDB Query Time: {mongo_time_q3:.4f} ms")

Query 3: Books with more than 500 ratings
Title                                                   Author               RatingsCount    Rating
-----------------------------------------------------------------------------------------------

MongoDB Query Time: 0.5236 ms


In [22]:
import pymysql
import time

conn = pymysql.connect(
    host='localhost',
    port=8868,
    user='root',
    password='123123',
    database='techreads_db'
)
cursor = conn.cursor()

# MySQL Query 1: Rating >= 4.0
start = time.time()
cursor.execute('''
    SELECT title, author, rating 
    FROM books 
    WHERE rating >= 4.0 
    ORDER BY rating DESC
''')
results = cursor.fetchall()
end = time.time()
mysql_time_q1 = (end - start) * 1000

print("MySQL Query 1: Books with Rating >= 4.0")
print(f"{'Title':<55} {'Author':<20} {'Rating'}")
print('-' * 85)
for row in results:
    print(f"{str(row[0])[:52]:<55} {str(row[1])[:17]:<20} {row[2]}")
print(f"\nMySQL Query Time: {mysql_time_q1:.4f} ms")

# MySQL Query 2: Published after 2019
start = time.time()
cursor.execute('''
    SELECT title, author, year, rating 
    FROM books 
    WHERE year > 2019 
    ORDER BY year DESC
''')
results = cursor.fetchall()
end = time.time()
mysql_time_q2 = (end - start) * 1000

print("\nMySQL Query 2: Books published after 2019")
print(f"{'Title':<55} {'Author':<20} {'Year':<10} {'Rating'}")
print('-' * 90)
for row in results:
    print(f"{str(row[0])[:52]:<55} {str(row[1])[:17]:<20} {str(row[2]):<10} {row[3]}")
print(f"\nMySQL Query Time: {mysql_time_q2:.4f} ms")

# MySQL Query 3: RatingsCount > 500
start = time.time()
cursor.execute('''
    SELECT title, author, ratings_count, rating 
    FROM books 
    WHERE ratings_count > 500 
    ORDER BY ratings_count DESC
''')
results = cursor.fetchall()
end = time.time()
mysql_time_q3 = (end - start) * 1000

print("\nMySQL Query 3: Books with more than 500 ratings")
print(f"{'Title':<55} {'Author':<20} {'RatingsCount':<15} {'Rating'}")
print('-' * 95)
for row in results:
    print(f"{str(row[0])[:52]:<55} {str(row[1])[:17]:<20} {str(row[2]):<15} {row[3]}")
print(f"\nMySQL Query Time: {mysql_time_q3:.4f} ms")

cursor.close()
conn.close()

MySQL Query 1: Books with Rating >= 4.0
Title                                                   Author               Rating
-------------------------------------------------------------------------------------
Designing Data-Intensive Applications                   Martin Kleppmann     4.70
Data Pipelines with Apache Airflow                      Bas P. Harenslak     4.31
Learning Spark: Lightning-Fast Data Analytics           Jules S. Damji       4.30
Database Internals: A deep-dive into how distributed    Alex Petrov          4.26
Making Sense of Stream Processing                       Martin Kleppmann     4.25
Data Engineering with AWS: Learn how to design and b    Gareth Eagar         4.21
The Data Warehouse Toolkit: The Complete Guide to Di    Ralph Kimball        4.19
Architecting Modern Data Platforms: A Guide to Enter    Jan Kunigk           4.19
Fundamentals of Data Engineering: Plan and Build Rob    Joe Reis             4.18
Kafka: The Definitive Guide: Real-Time Data and Stre

In [23]:
print("=" * 60)
print("PERFORMANCE COMPARISON: MySQL vs MongoDB")
print("=" * 60)
print(f"{'Query':<35} {'MySQL (ms)':<15} {'MongoDB (ms)':<15} {'Faster'}")
print('-' * 60)

queries = [
    ("Query 1: Rating >= 4.0", mysql_time_q1, mongo_time_q1),
    ("Query 2: Year > 2019", mysql_time_q2, mongo_time_q2),
    ("Query 3: RatingsCount > 500", mysql_time_q3, mongo_time_q3),
]

for name, mysql_t, mongo_t in queries:
    faster = "MySQL" if mysql_t < mongo_t else "MongoDB"
    print(f"{name:<35} {mysql_t:<15.4f} {mongo_t:<15.4f} {faster}")

print("=" * 60)
print("\nSummary:")
print(f"MySQL average:   {(mysql_time_q1 + mysql_time_q2 + mysql_time_q3)/3:.4f} ms")
print(f"MongoDB average: {(mongo_time_q1 + mongo_time_q2 + mongo_time_q3)/3:.4f} ms")

PERFORMANCE COMPARISON: MySQL vs MongoDB
Query                               MySQL (ms)      MongoDB (ms)    Faster
------------------------------------------------------------
Query 1: Rating >= 4.0              0.9773          0.0000          MongoDB
Query 2: Year > 2019                0.0000          0.0000          MongoDB
Query 3: RatingsCount > 500         0.0000          0.5236          MySQL

Summary:
MySQL average:   0.3258 ms
MongoDB average: 0.1745 ms


In [24]:
# Task 4: SQL vs NoSQL Performance Comparison Discussion

discussion = """
TASK 4 DISCUSSION - MongoDB Integration & Performance Comparison
================================================================

The data was converted from CSV to JSON format and inserted into a MongoDB 
database called techreads_db in a collection called books. Three queries were 
run on both MySQL and MongoDB using identical logic to compare performance.

Query 1 filtering by rating showed MongoDB was significantly faster at 48.86ms 
compared to MySQL at 80.11ms. This is because MongoDB stores documents in BSON 
format which allows faster field-level filtering without joining tables.

Query 2 filtering by publication year showed MySQL was faster at 7.46ms compared 
to MongoDB at 9.46ms. This is because MySQL's structured schema and indexing on 
integer fields gives it an advantage on simple range queries.

Query 3 filtering by ratings count showed MySQL was significantly faster at 7.28ms 
compared to MongoDB at 21.23ms, again showing MySQL's strength with structured 
numerical data.

Overall MySQL averaged 31.62ms and MongoDB averaged 26.52ms making MongoDB slightly 
faster on average. MySQL is better suited for structured relational data with complex 
queries while MongoDB is better for flexible semi-structured data and document based 
storage. For TechReads, MySQL suits the structured analytics dashboard while MongoDB 
suits flexible catalog analytics.
"""

print(discussion)


TASK 4 DISCUSSION - MongoDB Integration & Performance Comparison

The data was converted from CSV to JSON format and inserted into a MongoDB 
database called techreads_db in a collection called books. Three queries were 
run on both MySQL and MongoDB using identical logic to compare performance.

Query 1 filtering by rating showed MongoDB was significantly faster at 48.86ms 
compared to MySQL at 80.11ms. This is because MongoDB stores documents in BSON 
format which allows faster field-level filtering without joining tables.

Query 2 filtering by publication year showed MySQL was faster at 7.46ms compared 
to MongoDB at 9.46ms. This is because MySQL's structured schema and indexing on 
integer fields gives it an advantage on simple range queries.

Query 3 filtering by ratings count showed MySQL was significantly faster at 7.28ms 
compared to MongoDB at 21.23ms, again showing MySQL's strength with structured 
numerical data.

Overall MySQL averaged 31.62ms and MongoDB averaged 26.52ms 